# Overview

Goal of this notebook is to create the train/test/holdout splits for the project.

For the splits themselves, we want to pull one row per building. In this data basically every building shows up multiple times, so we only want a single instance of that building to train/test on.

In [53]:
import polars as pl
from src.helpers.eda import get_column_value_frequency

data_directory = "../../data/energy"
df = pl.read_parquet(f"{data_directory}/s_energy.parquet")

In [54]:
df

OSEBuildingID,DataYear,BuildingName,BuildingType,Neighborhood,YearBuilt,NumberofFloors,NumberofBuildings,PropertyGFABuildings,EPAPropertyType,EUI,EUI_WN,Year2025,DataYears,NumYears
i64,i64,str,str,str,i64,i64,i64,i64,str,f32,f32,bool,list[i64],u32
2,2025,"""Paramount Hotel""","""NonResidential""","""EAST""",1996,11,1,88502,"""Hotel""",69.300003,69.0,true,"[2015, 2016, … 2025]",11
3,2025,"""Westin Hotel (Parent Building)""","""NonResidential""","""LAKE UNION""",1969,41,3,759392,"""Hotel""",89.900002,89.900002,true,"[2015, 2016, … 2025]",11
5,2025,"""Hotel Max""","""NonResidential""","""LAKE UNION""",1926,10,1,61320,"""Hotel""",83.599998,83.199997,true,"[2015, 2016, … 2025]",11
8,2025,"""Warwick Seattle Hotel""","""NonResidential""","""LAKE UNION""",1980,18,1,113580,"""Hotel""",86.099998,85.699997,true,"[2015, 2016, … 2025]",11
9,2025,"""West Precinct (Seattle Police)""","""NonResidential""","""LAKE UNION""",1999,2,1,60090,"""Police Station""",172.100006,171.699997,true,"[2015, 2016, … 2025]",11
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
50038,2015,"""Salmon Bay Marine Center West""","""NonResidential""","""MAGNOLIA / QUEEN ANNE""",2014,2,1,25532,"""Mixed Use Property""",26.799999,29.1,false,"[2015, 2016, … 2025]",11
50049,2015,"""PACIFIC CENTER CONDOMINIUM""","""Multifamily LR (1-4)""","""DOWNTOWN""",2000,4,1,44650,"""Multifamily LR (1-4)""",25.6,27.799999,false,"[2015, 2016, … 2025]",11
50055,2015,"""NORA (IDENTITY APTS 4123)""","""Multifamily MR (5-9)""","""NORTHEAST""",2014,7,1,42393,"""Multifamily MR (5-9)""",48.599998,55.099998,false,"[2015, 2016, … 2025]",11


## Set it all up

Plan:

In [ ]:
final_year_set = df.filter(pl.col("DataYear") == 2025)

buildings = (
    final_year_set
    .select("OSEBuildingID")
    .unique()
    .shape[0]
)
print(f"\nNumber of Buildings in Dataset: {buildings}\n")

In [ ]:
# Goal:
df_shuffled = (
    df
    .filter(
        pl.int_range(0, pl.len())
        .shuffle(seed=101)
        .over("OSEBuildingID") == 0
    )
)

df_shuffled

In [ ]:
(
    df_shuffled
    .group_by("OSEBuildingID")
    .len()
    .sort("len", descending=True)
)

In [ ]:
get_column_value_frequency(df_shuffled, "DataYear")